In [45]:
from discovery_child_development.getters import gtr
import pandas as pd

import datetime

import importlib
importlib.reload(gtr);

from discovery_child_development import PROJECT_DIR
ENRICHED_DATA_DIR = PROJECT_DIR / 'outputs/enrichments'

In [7]:
projects_df = gtr.get_gtr_from_s3(table='projects')

In [40]:
funds_dict = gtr.get_gtr_from_s3(table='funds')

In [19]:
projects_df[0].keys()

dict_keys(['links', 'id', 'href', 'created', 'identifiers', 'title', 'status', 'grantCategory', 'leadFunder', 'leadOrganisationDepartment', 'abstractText', 'techAbstractText', 'healthCategories', 'researchActivities', 'researchSubjects', 'researchTopics', 'rcukProgrammes'])

In [104]:
funds_id = []
start = []
ends = []
value = []
currencies = []

for funds in funds_dict:
    funds_id.append(funds['id'])
    start.append(funds['start'])
    ends.append(funds['end'])
    value.append(funds['valuePounds']['amount'] if 'valuePounds' in funds else None)
    currencies.append(funds['valuePounds']['currencyCode'] if 'valuePounds' in funds else None)

funds_df = (
    pd.DataFrame({'funds_id': funds_id, 'start': start, 'ends': ends, 'amount': value, 'currency': currencies})
    .assign(start = lambda df: df.start.apply(lambda x: datetime.datetime.fromtimestamp(int(x)/1000).strftime('%Y-%m-%d')))
    .assign(ends = lambda df: df.ends.apply(lambda x: datetime.datetime.fromtimestamp(int(x)/1000).strftime('%Y-%m-%d')))
)


In [105]:
funds_df.head(1)

,funds_id,start,ends,amount,currency
0,2293906D-CC22-4E09-A91E-CE2CC6FE6CCD,2022-08-01,2026-07-31,1529795,GBP


In [108]:
ids = []
titles = []
abstracts = []
tech_abstracts = []
starts = []
ends = []
funds_id = []

for project in projects_df:
    ids.append(project['id'])
    titles.append(project['title']) if 'title' in project.keys() else titles.append('')
    abstracts.append(project['abstractText']) if 'abstractText' in project.keys() else abstracts.append('')
    tech_abstracts.append(project['techAbstractText']) if 'techAbstractText' in project.keys() else tech_abstracts.append('')

    funds = [l for l in project['links']['link'] if l['rel'] == "FUND"][0]
    starts.append(datetime.datetime.fromtimestamp(int(funds['start'])/1000).strftime('%Y-%m-%d'))
    ends.append(datetime.datetime.fromtimestamp(int(funds['end'])/1000).strftime('%Y-%m-%d'))
    funds_id.append(funds['href'].split('/')[-1])

gtr_texts_df = (
    pd.DataFrame({
        'id': ids,
        'title': titles,
        'abstract': abstracts,
        'tech_abstract': tech_abstracts,
        'start': starts,
        "end": ends,
        "funds_id": funds_id,
    })
    .assign(text = lambda x: x['title'] + '. ' + x['abstract'] + '. ' + x['tech_abstract'])
    .merge(funds_df[['funds_id', 'amount', 'currency']], on='funds_id', how='left', suffixes=('', '_funds'))
)


In [109]:
gtr_texts_df.head(10)

,id,title,abstract,tech_abstract,start,end,funds_id,text,amount,currency
0,9FCC3406-8050-4D4B-A4D6-14ABC3A9AC51,20-BBSRC/NSF-BIO Regulatory control of innate immune response in marine invertebrates,"Animals live in constantly changing environments that are rich in microbial life. Immune systems orchestrate the complex, dynamic relationships between animals and microbes by both eliminating pathogenic microbes as well as promoting the growth of beneficial microbiota. However, our understanding of innate immune mechanisms in most animal phyla, including vertebrates, remains limited. The proposed research addresses this considerable knowledge gap by exploiting the experimental advantages of echinoderm larvae to investigate immune responses in a relatively unexplored animal lineage. Echinoderm larvae are morphologically simple, free-swimming multicellular organisms equipped with a sophisticated cellular and molecular immune system. By integrating recently available sequencing data from several echinoderm species with technology for measuring gene expression at single-cell resolution, the proposed research will significantly advance our understanding of animal immunity. This proposa...","Animals live in constantly changing environments that are rich in microbial life. Immune systems orchestrate the complex, dynamic relationships between animals and microbes by both eliminating pathogenic microbes as well as promoting the growth of beneficial microbiota. However, our understanding of innate immune mechanisms in most animal phyla remains limited. The proposed research will address this knowledge gap by exploiting the experimental advantages of echinoderm larvae to investigate immune responses.\nThe proposed work will investigate immunity in four echinoderm species and characterising the response to both bacterial and viral pathogens. The echinoderm species were selected to reflect a range of taxonomic distances, ecological niches, and life history traits. The proposed work will be carried out in three aims. First, larval immune cell populations will be defined using in vivo microscopy and functional assays. Second, the change in gene activity in response to immune ch...",2022-01-01,2024-12-31,8146E9B5-D152-4584-8B7A-061CAF34D028,"20-BBSRC/NSF-BIO Regulatory control of innate immune response in marine invertebrates. Animals live in constantly changing environments that are rich in microbial life. Immune systems orchestrate the complex, dynamic relationships between animals and microbes by both eliminating pathogenic microbes as well as promoting the growth of beneficial microbiota. However, our understanding of innate immune mechanisms in most animal phyla, including vertebrates, remains limited. The proposed research addresses this considerable knowledge gap by exploiting the experimental advantages of echinoderm larvae to investigate immune responses in a relatively unexplored animal lineage. Echinoderm larvae are morphologically simple, free-swimming multicellular organisms equipped with a sophisticated cellular and molecular immune system. By integrating recently available sequencing data from several echinoderm species with technology for measuring gene expression at single-cell resolution, the proposed...",513195,GBP
1,A0987686-9E04-4618-8588-162133A29DB8,Design and development of a Virtual Standardised Patient platform for improving medical training,"Abstracts are not currently available in GtR for all funded research. This is normally because the abstract was not required at the time of proposal submission, but may be because it included sensitive information such as personal details.",,2020-01-01,2021-06-30,BE7A902A-33CA-4FCF-844B-BCE32EC53DC9,"Design and development of a Virtual Standardised Patient platform for improving medical training. Abstracts are not currently available in GtR for all funded research. This is normally because the abstract was not required at the time of proposal submission, but may be because it includ

In [114]:
gtr_texts_df.drop(['abstract', 'tech_abstract'], axis=1).query("start >= '2013-01-01'").to_csv(ENRICHED_DATA_DIR / 'gtr_texts.csv', index=False)

In [112]:
gtr_texts_df.head(5)

,id,title,abstract,tech_abstract,start,end,funds_id,text,amount,currency
0,9FCC3406-8050-4D4B-A4D6-14ABC3A9AC51,20-BBSRC/NSF-BIO Regulatory control of innate immune response in marine invertebrates,"Animals live in constantly changing environments that are rich in microbial life. Immune systems orchestrate the complex, dynamic relationships between animals and microbes by both eliminating pathogenic microbes as well as promoting the growth of beneficial microbiota. However, our understanding of innate immune mechanisms in most animal phyla, including vertebrates, remains limited. The proposed research addresses this considerable knowledge gap by exploiting the experimental advantages of echinoderm larvae to investigate immune responses in a relatively unexplored animal lineage. Echinoderm larvae are morphologically simple, free-swimming multicellular organisms equipped with a sophisticated cellular and molecular immune system. By integrating recently available sequencing data from several echinoderm species with technology for measuring gene expression at single-cell resolution, the proposed research will significantly advance our understanding of animal immunity. This proposa...","Animals live in constantly changing environments that are rich in microbial life. Immune systems orchestrate the complex, dynamic relationships between animals and microbes by both eliminating pathogenic microbes as well as promoting the growth of beneficial microbiota. However, our understanding of innate immune mechanisms in most animal phyla remains limited. The proposed research will address this knowledge gap by exploiting the experimental advantages of echinoderm larvae to investigate immune responses.\nThe proposed work will investigate immunity in four echinoderm species and characterising the response to both bacterial and viral pathogens. The echinoderm species were selected to reflect a range of taxonomic distances, ecological niches, and life history traits. The proposed work will be carried out in three aims. First, larval immune cell populations will be defined using in vivo microscopy and functional assays. Second, the change in gene activity in response to immune ch...",2022-01-01,2024-12-31,8146E9B5-D152-4584-8B7A-061CAF34D028,"20-BBSRC/NSF-BIO Regulatory control of innate immune response in marine invertebrates. Animals live in constantly changing environments that are rich in microbial life. Immune systems orchestrate the complex, dynamic relationships between animals and microbes by both eliminating pathogenic microbes as well as promoting the growth of beneficial microbiota. However, our understanding of innate immune mechanisms in most animal phyla, including vertebrates, remains limited. The proposed research addresses this considerable knowledge gap by exploiting the experimental advantages of echinoderm larvae to investigate immune responses in a relatively unexplored animal lineage. Echinoderm larvae are morphologically simple, free-swimming multicellular organisms equipped with a sophisticated cellular and molecular immune system. By integrating recently available sequencing data from several echinoderm species with technology for measuring gene expression at single-cell resolution, the proposed...",513195,GBP
1,A0987686-9E04-4618-8588-162133A29DB8,Design and development of a Virtual Standardised Patient platform for improving medical training,"Abstracts are not currently available in GtR for all funded research. This is normally because the abstract was not required at the time of proposal submission, but may be because it included sensitive information such as personal details.",,2020-01-01,2021-06-30,BE7A902A-33CA-4FCF-844B-BCE32EC53DC9,"Design and development of a Virtual Standardised Patient platform for improving medical training. Abstracts are not currently available in GtR for all funded research. This is normally because the abstract was not required at the time of proposal submission, but may be because it includ

## Check relevant gtr texts

In [88]:
from nesta_ds_utils.loading_saving import S3
from discovery_child_development import S3_BUCKET

In [89]:
gtr_texts_relevant_df = S3.download_obj(S3_BUCKET, f"data/outputs/binary_classifier/gtr_texts_relevance_labelled.csv", "dataframe")

In [94]:
gtr_texts_relevant_df.head(1)

,Unnamed: 0,id,start,end,text,predictions
0,0,9FCC3406-8050-4D4B-A4D6-14ABC3A9AC51,2022-01-01,2024-12-31,"20-BBSRC/NSF-BIO Regulatory control of innate immune response in marine invertebrates. Animals live in constantly changing environments that are rich in microbial life. Immune systems orchestrate the complex, dynamic relationships between animals and microbes by both eliminating pathogenic microbes as well as promoting the growth of beneficial microbiota. However, our understanding of innate immune mechanisms in most animal phyla, including vertebrates, remains limited. The proposed research addresses this considerable knowledge gap by exploiting the experimental advantages of echinoderm larvae to investigate immune responses in a relatively unexplored animal lineage. Echinoderm larvae are morphologically simple, free-swimming multicellular organisms equipped with a sophisticated cellular and molecular immune system. By integrating recently available sequencing data from several echinoderm species with technology for measuring gene expression at single-cell resolution, the proposed...",0


In [93]:
pd.set_option("max_colwidth", 1000)
gtr_texts_relevant_df.query("predictions == 1").to_csv(ENRICHED_DATA_DIR / 'gtr_texts_relevant.csv', index=False)